# Обработка исключений

основной механизим обработки исключений - это `try/except/finally`

In [2]:
ans = 92
try:
    if ans != 42:
        raise ValueError("Wrong answer")
except RecursionError as e:
    print(f"Error {e}")
    raise e
finally:
    print("cleanup")

cleanup


ValueError: Wrong answer

- `try` мы пишем код, который может вызвать ошибку
- `except` ловит конкретную ошибку
- `else` будет выполнятся если в блоке `try` не возникло ошибок
- `finally` выполняется всегда

все эти блоки определяюся динамически

In [3]:
def outer():
    try:
        middle()
    except Exception as e:
        print(f"Exception: {e}")
        raise e

def middle():
    try:
        inner()
    finally:
        print("cleanup")

def inner():
    raise RecursionError("Lol")

outer()

cleanup
Exception: Lol


RecursionError: Lol

можно делать многоразовую обработку

In [4]:
try:
    pass
except (ValueError, ArithmeticError):
    pass
except TypeError as e:
    pass

### Иерархия исключений

In [5]:
BaseException.__subclasses__()

[Exception,
 GeneratorExit,
 SystemExit,
 KeyboardInterrupt,
 asyncio.exceptions.CancelledError,
 exceptiongroup.BaseExceptionGroup]

In [7]:
len(Exception.__subclasses__())

127

In [8]:
Exception.__subclasses__()[:10]

[TypeError,
 StopAsyncIteration,
 StopIteration,
 ImportError,
 OSError,
 EOFError,
 RuntimeError,
 NameError,
 AttributeError,
 SyntaxError]

чтобы ловить все исключения как раз таки и нужен Exception

In [9]:
import sys

try:
    sys.exit()
except Exception:
    pass

SystemExit: 

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### RuntimeError

Кидается когда что-то пошло не так во время выполнения

In [11]:
d = {"foo": 32, "bar": 34}
for key in d:
    d.pop(key)

RuntimeError: dictionary changed size during iteration

### ImportError

нужен для импортов

In [12]:
try:
    import foobar_speedups as foobar
except ImportError:
    import foobar_slow as foobar

ModuleNotFoundError: No module named 'foobar_slow'

### AtributeError

обращаемся к атрибуту, а его нет

In [13]:
class A:
    pass

A().x

AttributeError: 'A' object has no attribute 'x'

### LookupError

IndexError и KeyError - кидаются когда мы обратились по несуществующему индексу или по несуществующему значению

In [14]:
[][0]

IndexError: list index out of range

In [15]:
{}[0]

KeyError: 0

## TypeError

ошибка типов

In [16]:
[][None]

<>:1: SyntaxWarning: list indices must be integers or slices, not NoneType; perhaps you missed a comma?
<>:1: SyntaxWarning: list indices must be integers or slices, not NoneType; perhaps you missed a comma?
/var/folders/xn/ds850dk573bdzhzp8cdf80yh0000gn/T/ipykernel_30426/1135167476.py:1: SyntaxWarning: list indices must be integers or slices, not NoneType; perhaps you missed a comma?
  [][None]
/var/folders/xn/ds850dk573bdzhzp8cdf80yh0000gn/T/ipykernel_30426/1135167476.py:1: SyntaxWarning: list indices must be integers or slices, not NoneType; perhaps you missed a comma?
  [][None]
/var/folders/xn/ds850dk573bdzhzp8cdf80yh0000gn/T/ipykernel_30426/1135167476.py:1: SyntaxWarning: list indices must be integers or slices, not NoneType; perhaps you missed a comma?
  [][None]


TypeError: list indices must be integers or slices, not NoneType

### ValueError

когда проблемы со значениями

In [18]:
int("turbo")

ValueError: invalid literal for int() with base 10: 'turbo'

### Собственные исключения

In [20]:
def Error(Exception):
    pass

In [21]:
e = Exception("134", 4123434, "world")
e.args

('134', 4123434, 'world')

в Exception запоминается то, что мы туда передаем

объект traceback передает срез программы в момента падения

In [22]:
import traceback

def foo():
    bar()

def bar():
    raise Exception

try:
    foo()
except Exception as e:
    traceback.print_tb(e.__traceback__)

  File "/var/folders/xn/ds850dk573bdzhzp8cdf80yh0000gn/T/ipykernel_30426/1513588552.py", line 10, in <module>
    foo()
  File "/var/folders/xn/ds850dk573bdzhzp8cdf80yh0000gn/T/ipykernel_30426/1513588552.py", line 4, in foo
    bar()
  File "/var/folders/xn/ds850dk573bdzhzp8cdf80yh0000gn/T/ipykernel_30426/1513588552.py", line 7, in bar
    raise Exception


### raise

In [23]:
raise Exception("foo")

Exception: foo

In [25]:
e = Exception
raise Exception("foo") from e

Exception: foo

In [26]:
raise Exception("foo") from None

Exception: foo

## With 

with нужен для автоматического закрытия ресурсов

In [ ]:
with open("data.db") as db:
    with open(data2.db) as db2:
        pass
    # calls closde()
# calls close()

чтобы файл как можно раньше закрывать можно сделать так:

In [ ]:
with open("input.txt") as f:
    text = f.read()
process(text)

как раблотает with внутри:

In [ ]:
manager = acquire_resource()
r = manager.__enter__()
try:
    do_something(r)
finally:
    exc_type, exc_valut, tb = sys.exc_info()
    suppress = manager.__exit__(exc_type, exc_value, tb)
    if exc_value in not None and not suppress:
        raise exc_value

для использования with нужны методы `__enter__` и `__exit__` у класса